In [3]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 43.9 MB/s eta 0:00:00


In [35]:
nltk.download("wordnet")
nltk.download("punkt_tab")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
import numpy as np
import pandas as pd
import nltk
import gensim
import re
import tqdm

In [29]:
from google.colab import files
script = pd.read_csv("/content/cleaned_df_of_script.csv")

In [16]:
script.head(2)

,line_id,speaker,type,text
0,1,NARRATOR,scene_heading,INT. DELTA GAMMA HOUSE - DAY ...
1,2,NARRATOR,scene_description,"flock of abstract, silky, golden strands -- PU..."


### Cleaning the data frame

In [30]:
from tempfile import tempdir
# Cleaning the dataframe
script.head(2)
# DROPPING FEATURES
#script = script.drop(['line_id','type'],axis=1)
# FILTERING ONLY TWO SPEAKERS
script = script[script['speaker'].isin (["EMMETT","WARNER"])]
script = script.reset_index(drop=True)
# CLEANING THE TARGET FEATURE
script['speaker'] = (
    script['speaker']
    .astype(str)
    .str.strip()
    .str.replace(r'[^a-zA-Z]','',regex=True)
    .str.upper
)
# CLEANING THE INDEPENDENT FEATURE
script['text'] = (
    script['text']
    .astype(str)
    .str.strip()
    .str.replace(r'[^a-zA-Z]',' ',regex=True)
    .str.replace(r'\bcontinuing\b','',regex=True)
    .str.replace(r'\bINT\b','',regex=True)
    #.str.replace(r'\s+',' ',regex=True)
    .str.lower()
)

from nltk.stem import WordNetLemmatizer
nltk.download("stopwords")
from nltk.corpus import stopwords
lem = WordNetLemmatizer()
corpus = []
for i in range(0,len(script)):
  temp = script.loc[i,'text'].split()
  temp = [lem.lemmatize(word) for word in temp]
  temp = ' '.join(temp)
  corpus.append(temp)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [31]:
corpus[0]

'you re beautiful'

### Word2Vec

In [36]:
# WORD2VEC PREPROCESSING
from nltk import sent_tokenize
from gensim.utils import simple_preprocess
words = []
for cor in corpus:
  sent = sent_tokenize(cor)
  for s in sent:
    words.append(simple_preprocess(s))

In [40]:
len(words)

350

You have 350 sentences (rows) and each have a list of sentences and inside them they have list of words

In [37]:
# WORD2VEC MODEL
from gensim.models import Word2Vec
model = gensim.models.Word2Vec(words)

In [38]:
model.corpus_count

350

In [39]:
model.wv['warner'].shape

(100,)

### Average Word2Vec

In [41]:
def avg_word2vec(doc,model):
  vecs = [model.wv[word] for word in doc if word in model.wv.index_to_key]
  if len(vecs) == 0: #sometimes you may not find all the words in the sentence also present in the vocabulary of the model
    return np.zeros(model.vector_size)
  else:
    return np.mean(vecs,axis=0)

In [42]:
from tqdm import tqdm
X = []
for i in tqdm(range(len(words))):
  X.append(avg_word2vec(words[i],model))

100%|██████████| 350/350 [00:00<00:00, 21890.94it/s]


In [44]:
print(script.shape)
print(len(X))

(354, 4)
350


The four rows may be where the entire rows were special characters or zeros so we now need to find and delete their relevant target rows and we will be good to goooo

In [45]:
# Using the lambda fucntion to get the length of each row(sentence in the corpus)
# Map is used to map each value of the corpus list where each value will be one row to the lambda function
# Now, filter out the rows which do not have null values
# The map and lambda function creates a boolean mask here : [true, false, true, false]
# Now script[[True, false, true]] will filter the entire dataframe where the rows are True
y = script[list(map(lambda x: len(x)>0,corpus))]
y = pd.get_dummies(y['speaker'])
y = y.iloc[:,0].values

In [46]:
y.shape

(350,)

### Reshaping the final dataframe

In [49]:
X[0]

array([ 5.2783918e-04,  6.0801287e-03,  3.1208291e-03,  5.5466769e-03,
        4.5292964e-03, -6.6630673e-03,  5.9186656e-05,  6.8510137e-03,
       -4.0352396e-03, -5.5303057e-03,  2.3733433e-03, -3.1115692e-03,
       -3.8161522e-03,  2.2128709e-03,  4.8641418e-03,  5.9420359e-04,
        2.0489432e-03,  3.5917829e-04, -4.1811136e-03, -5.1009003e-03,
       -8.4560632e-04, -6.3617593e-03,  5.1196166e-03, -9.2469957e-03,
       -1.5589853e-03, -2.7334676e-03, -6.1408081e-03,  5.6124288e-03,
        8.9911383e-04,  3.5789195e-03,  3.6600665e-03, -6.0546510e-03,
        2.7073856e-04, -3.1432465e-03,  2.0394872e-03,  2.0640525e-03,
        3.9517232e-03,  2.9893185e-03,  7.2737830e-04,  1.5546246e-03,
       -1.8359954e-04, -4.3971632e-03, -3.6738678e-03,  3.6879319e-03,
       -2.3228601e-03,  2.7505592e-03,  7.5157844e-03, -7.9678686e-04,
       -3.7597347e-04,  5.4595303e-03,  5.4726489e-03, -7.6719942e-03,
        2.9895809e-03,  3.8203436e-03, -3.9745625e-03,  3.9767521e-03,
      

In [50]:
X = np.array(X)

In [51]:
X[0]

array([ 5.27839176e-04,  6.08012872e-03,  3.12082912e-03,  5.54667693e-03,
        4.52929642e-03, -6.66306727e-03,  5.91866556e-05,  6.85101375e-03,
       -4.03523957e-03, -5.53030567e-03,  2.37334333e-03, -3.11156921e-03,
       -3.81615222e-03,  2.21287087e-03,  4.86414181e-03,  5.94203593e-04,
        2.04894319e-03,  3.59178288e-04, -4.18111356e-03, -5.10090031e-03,
       -8.45606322e-04, -6.36175927e-03,  5.11961663e-03, -9.24699567e-03,
       -1.55898533e-03, -2.73346761e-03, -6.14080811e-03,  5.61242877e-03,
        8.99113831e-04,  3.57891945e-03,  3.66006652e-03, -6.05465099e-03,
        2.70738557e-04, -3.14324652e-03,  2.03948724e-03,  2.06405250e-03,
        3.95172322e-03,  2.98931845e-03,  7.27378298e-04,  1.55462464e-03,
       -1.83599535e-04, -4.39716317e-03, -3.67386779e-03,  3.68793192e-03,
       -2.32286006e-03,  2.75055924e-03,  7.51578435e-03, -7.96786859e-04,
       -3.75973468e-04,  5.45953028e-03,  5.47264889e-03, -7.67199416e-03,
        2.98958085e-03,  

In [52]:
df_final = []
for i in range(len(X)):
  df_final.append(pd.DataFrame(X[i].reshape(1,-1))) # We have 350 rows and each rows contained 100 length vector reps which we converted into cols as they are the features, i,e average of the vector rep of the words
df = pd.concat(df_final, ignore_index=True) # Old index ignored and new index assigned

In [54]:
df.shape

(350, 100)

In [57]:
! pip install scikit-learn

In [59]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df,y,random_state=43,test_size=0.3)

In [61]:
print(f"The shape of training data is, X : {X_train.shape}, Y: {y_train.shape}")
print(f"The shape of test data is, X : {X_test.shape}, Y: {y_test.shape}")


The shape of training data is, X : (245, 100), Y: (245,)
The shape of test data is, X : (105, 100), Y: (105,)


now randomforest or whatever model we need can be built

*sarvam krishnarpanam*